In [1]:
import sys
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)

import numpy as np
import torch
import time
import json
from pathlib import Path
from typing import List

from GEMS_TCO import kernels_vecchia_cauchy   # new Cauchy kernel
from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed

## Load daily data (max-min ordering)

In [2]:
space         = ['1', '1']
lat_lon_resolution = [int(s) for s in space]
mm_cond_number: int = 100
years         = ['2025']
month_range   = [7]

output_path   = Path(config.mac_estimates_day_path)
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

lat_range_input = [-3, 2]
lon_range_input = [121, 131]

df_map, ord_mm, nns_map, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=lat_lon_resolution,
    mm_cond_number=mm_cond_number,
    years_=years,
    months_=month_range,
    lat_range=lat_range_input,
    lon_range=lon_range_input,
    is_whittle=False,
)

--- Global Monthly Mean for 2025-7: 241.4412 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---


In [3]:
daily_hourly_maps_vecc = []

for day_index in range(31):
    hour_indices = [day_index * 8, (day_index + 1) * 8]
    day_hourly_map, _ = data_load_instance.load_working_data(
        df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=torch.float64, keep_ori=True,
    )
    daily_hourly_maps_vecc.append(day_hourly_map)

print(f"Loaded {len(daily_hourly_maps_vecc)} days.")

Loaded 31 days.


## Model settings

### Cauchy kernel
```
C(d) = σ² / (1 + d / range_lon)^β
```
- **`gc_beta`** (β): polynomial decay exponent. `β=1` → standard Cauchy. Default 1.0.
- All other parameters identical to the Matérn model.

### Why Cauchy vs Matérn?
| | Matérn (ν=0.5) | Cauchy (β=1) |
|---|---|---|
| Decay | `exp(−d/α)` — exponential | `1/(1+d/α)` — polynomial |
| Long-range | negligible past ~3α | polynomial tail, never zero |
| Small α | heads ≈ uncorrelated | heads still capture tail → **more benefit from heads** |

In [4]:
v = 0.5   # Matern smoothness (kept for base class compatibility)

# ── Cauchy-specific ─────────────────────────────────────────────────────────
gc_beta = 1.0   # β: polynomial decay exponent  (try 0.5, 1.0, 2.0)

# ── Vecchia conditioning ─────────────────────────────────────────────────────
limit_A      = 8
limit_B      = 8
limit_C      = 8
nheads       = 300
daily_stride = 8   # >= 8 → Set C inactive (t and t-1 only)

# ── L-BFGS ──────────────────────────────────────────────────────────────────
LBFGS_LR         = 1.0
LBFGS_MAX_STEPS  = 3
LBFGS_HISTORY    = 100
LBFGS_MAX_EVAL   = 100

## Fit Cauchy Vecchia

In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"gc_beta: {gc_beta}")

days_list = [24]   # 0-based day index

all_results = {}

for day_idx in days_list:
    print(f'\n{"="*50}')
    print(f'  Day {day_idx+1}  (2024-07-{day_idx+1})  |  Cauchy β={gc_beta}')
    print(f'{"="*50}')

    daily_hourly_map_vecc = daily_hourly_maps_vecc[day_idx]

    # ── Initial values (same as Matern notebook) ──────────────────────────────
    init_sigmasq    = 13.059
    init_range_lat  = 0.2
    init_range_lon  = 0.25
    init_range_time = 1.5
    init_advec_lat  = 0.0218
    init_advec_lon  = -0.1689
    init_nugget     = 0.247

    init_phi2 = 1.0 / init_range_lon
    init_phi1 = init_sigmasq * init_phi2
    init_phi3 = (init_range_lon / init_range_lat)  ** 2
    init_phi4 = (init_range_lon / init_range_time) ** 2

    initial_vals = [
        np.log(init_phi1), np.log(init_phi2), np.log(init_phi3),
        np.log(init_phi4), init_advec_lat, init_advec_lon, np.log(init_nugget)
    ]
    params_list = [
        torch.tensor([val], requires_grad=True, dtype=torch.float64, device=DEVICE)
        for val in initial_vals
    ]

    # ── Build Cauchy model ────────────────────────────────────────────────────
    model = kernels_vecchia_cauchy.fit_cauchy_vecchia_lbfgs(
        smooth=v,
        gc_beta=gc_beta,
        input_map=daily_hourly_map_vecc,
        nns_map=nns_map,
        mm_cond_number=mm_cond_number,
        nheads=nheads,
        limit_A=limit_A, limit_B=limit_B, limit_C=limit_C,
        daily_stride=daily_stride,
    )
    model.precompute_conditioning_sets()

    optimizer = model.set_optimizer(
        params_list, lr=LBFGS_LR,
        max_iter=LBFGS_MAX_EVAL, history_size=LBFGS_HISTORY,
    )

    t0 = time.time()
    out, steps = model.fit_vecc_lbfgs(
        params_list, optimizer,
        max_steps=LBFGS_MAX_STEPS, grad_tol=1e-7,
    )
    elapsed = time.time() - t0
    print(f"Finished in {elapsed:.1f}s  ({steps+1} steps)")

    all_results[day_idx] = {'raw': out, 'elapsed': elapsed, 'gc_beta': gc_beta}

Device : cpu
gc_beta: 1.0

  Day 25  (2024-07-25)  |  Cauchy β=1.0
🚀 Pre-computing 3-group Vecchia [A=8, AB=17, ABC=26, stored=1]... [Mean Lat: -0.4817] [Set C: False] ✅ Done. (Heads: 2371, Tails A/AB/ABC: 17610/123776/0)
--- Starting Cauchy Vecchia L-BFGS  (β=1.0) ---
--- Step 1/3 / Loss: 1.388946 ---
  Param 0: Value=4.2341, Grad=0.026100170927931904
  Param 1: Value=2.0080, Grad=-0.01900263946789872
  Param 2: Value=0.7843, Grad=0.00081745244156064
  Param 3: Value=-4.5572, Grad=9.596455126209557e-05
  Param 4: Value=0.0057, Grad=0.1200447298216365
  Param 5: Value=-0.0101, Grad=-0.048741570047453275
  Param 6: Value=-0.3617, Grad=0.0010725948559091472
  Max Abs Grad: 1.200447e-01
------------------------------
--- Step 2/3 / Loss: 1.340265 ---
  Param 0: Value=3.4931, Grad=-7.4590166540501254e-06
  Param 1: Value=1.4595, Grad=5.088217222736044e-06
  Param 2: Value=0.8995, Grad=-8.62019213985586e-07
  Param 3: Value=-4.3215, Grad=-4.706736832706007e-07
  Param 4: Value=-0.0002, Grad

## Compare β values on one day

In [13]:
days_list = np.arange(0, 31)  # 0-based index
for day_idx in days_list:
    # Run the same day with multiple β values and compare losses + estimates
    beta_compare = [0.5, 1.0, 2.0]

    daily_hourly_map_vecc = daily_hourly_maps_vecc[day_idx]
    compare_results = {}

    for beta in beta_compare:
        print(f'\n--- β = {beta} ---')

        init_phi2 = 1.0 / 0.25
        init_phi1 = 13.059 * init_phi2
        init_phi3 = (0.25 / 0.2)  ** 2
        init_phi4 = (0.25 / 1.5) ** 2
        initial_vals = [
            np.log(init_phi1), np.log(init_phi2), np.log(init_phi3),
            np.log(init_phi4), 0.0218, -0.1689, np.log(0.247)
        ]
        params_list = [
            torch.tensor([val], requires_grad=True, dtype=torch.float64, device=DEVICE)
            for val in initial_vals
        ]

        model = kernels_vecchia_cauchy.fit_cauchy_vecchia_lbfgs(
            smooth=v, gc_beta=beta,
            input_map=daily_hourly_map_vecc,
            nns_map=nns_map, mm_cond_number=mm_cond_number,
            nheads=nheads,
            limit_A=limit_A, limit_B=limit_B, limit_C=limit_C,
            daily_stride=daily_stride,
        )
        model.precompute_conditioning_sets()

        opt = model.set_optimizer(params_list, lr=LBFGS_LR,
                                max_iter=LBFGS_MAX_EVAL, history_size=LBFGS_HISTORY)
        t0  = time.time()
        out, steps = model.fit_vecc_lbfgs(params_list, opt,
                                        max_steps=LBFGS_MAX_STEPS, grad_tol=1e-7)
        compare_results[beta] = {'raw': out, 'elapsed': time.time() - t0}

    print('\n=== β comparison (final loss) ===')
    for beta, r in compare_results.items():
        print(f'  β={beta}  loss={r["raw"][-1]:.6f}  ({r["elapsed"]:.1f}s)')


--- β = 0.5 ---
🚀 Pre-computing 3-group Vecchia [A=8, AB=17, ABC=26, stored=1]... [Mean Lat: -0.4855] [Set C: False] ✅ Done. (Heads: 2184, Tails A/AB/ABC: 17773/111471/0)
--- Starting Cauchy Vecchia L-BFGS  (β=0.5) ---
--- Step 1/3 / Loss: 1.519568 ---
  Param 0: Value=4.9898, Grad=6.8365202780025925e-06
  Param 1: Value=2.1884, Grad=-1.9510915854331075e-06
  Param 2: Value=0.3817, Grad=1.670515469172747e-06
  Param 3: Value=-3.1947, Grad=4.1095264213894296e-07
  Param 4: Value=-0.0048, Grad=1.5195260898350632e-06
  Param 5: Value=-0.1843, Grad=8.571423054624938e-07
  Param 6: Value=-2.4639, Grad=1.21257919015003e-07
  Max Abs Grad: 6.836520e-06
------------------------------
--- Step 2/3 / Loss: 1.363092 ---
  Param 0: Value=4.9898, Grad=6.8365202780025925e-06
  Param 1: Value=2.1884, Grad=-1.9510915854331075e-06
  Param 2: Value=0.3817, Grad=1.670515469172747e-06
  Param 3: Value=-3.1947, Grad=4.1095264213894296e-07
  Param 4: Value=-0.0048, Grad=1.5195260898350632e-06
  Param 5: Va

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

  β=0.5  loss=1.363092  (55.7s)
  β=1.0  loss=1.363661  (40.3s)
  β=2.0  loss=1.364855  (42.2s)

  === β comparison (final loss) ===
  β=0.5  loss=1.550194  (51.8s)
  β=1.0  loss=1.549929  (51.2s)
  β=2.0  loss=1.550553  (54.9s)

  === β comparison (final loss) ===
  β=0.5  loss=1.516808  (63.4s)
  β=1.0  loss=1.516806  (49.3s)
  β=2.0  loss=1.517450  (56.2s)

  === β comparison (final loss) ===
  β=0.5  loss=1.453112  (54.2s)
  β=1.0  loss=1.454397  (51.1s)
  β=2.0  loss=1.455976  (45.6s)

  === β comparison (final loss) ===
  β=0.5  loss=1.496065  (54.2s)
  β=1.0  loss=1.495891  (51.8s)
  β=2.0  loss=1.496536  (49.6s)

  === β comparison (final loss) ===
  β=0.5  loss=1.482535  (57.2s)
  β=1.0  loss=1.482669  (50.8s)
  β=2.0  loss=1.483244  (51.2s)

  === β comparison (final loss) ===
  β=0.5  loss=1.478072  (54.9s)
  β=1.0  loss=1.478519  (46.7s)
  β=2.0  loss=1.479530  (47.4s)

  === β comparison (final loss) ===
  β=0.5  loss=1.485135  (68.4s)
  β=1.0  loss=1.485325  (43.2s)
  β=2.0  loss=1.486718  (42.5s)

## Save results to JSON

In [ ]:
output_path.mkdir(parents=True, exist_ok=True)

for day_idx, res in all_results.items():
    raw   = res['raw']
    beta  = res['gc_beta']
    fname = output_path / f"cauchy_b{beta}_2024_07_{day_idx+1:02d}.json"
    payload = {
        'model':     'cauchy_vecchia',
        'gc_beta':   beta,
        'year':      2024,
        'month':     7,
        'day':       day_idx + 1,
        'raw_params': raw[:7],
        'final_loss': raw[7],
        'elapsed_s':  res['elapsed'],
    }
    with open(fname, 'w') as f:
        json.dump(payload, f, indent=2)
    print(f'Saved: {fname}')